# SpaceRec C2 xDBiT workflow with OpenPhenom

This notebook follows the same public API sequence as `spacerec.ipynb`. It reuses the supplied C2 deconvolution and routes every cerebellum call to the validated workflow that produced `resources/cerebellum`; no approximate or alternate implementation is used.

| Step | API | Output |
| --- | --- | --- |
| 1 | `spacerec.rctd_ref(...)` | platform-normalized `mu_ref` and gene list |
| 2 | `spacerec.ge(...)`, `spacerec.mask(...)` | OpenPhenom embedding and filtered tissue grids |
| 3.1 | `spacerec.stage1(...)` | router checkpoint and grid types |
| 3.2 | `spacerec.stage2(...)` | expression checkpoint and grid predictions |
| 4 | display | requested C2 window images |

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from pprint import pprint

import anndata as ad
import numpy as np
import pandas as pd


def find_spacerec_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        for candidate in (path, path / 'external' / 'SpaceRec'):
            if (candidate / 'spacerec' / 'api.py').is_file():
                return candidate
    raise FileNotFoundError('Could not find the repository root containing spacerec/api.py')


def require_files(*paths: Path) -> None:
    missing = [str(path) for path in paths if not path.is_file() or path.stat().st_size == 0]
    if missing:
        raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))


ROOT = find_spacerec_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import spacerec.api as spacerec

dataset = 'cerebellum'
sample_id = 'C2'
resource_root = ROOT / 'resources' / dataset
sample_root = resource_root / sample_id
deconv_csv = sample_root / 'deconv' / 'deconv.csv'
stage1_st_h5ad = sample_root / 'dbit' / 'st_stage1.h5ad'
stage1_gene_list = sample_root / 'reference' / 'stage1_placeholder_genes.txt'
fullres_image = sample_root / 'image' / 'fullres_4ch.ome.tif'
spot_positions = sample_root / 'dbit' / 'tissue_positions.csv'
require_files(deconv_csv, stage1_st_h5ad, stage1_gene_list, fullres_image, spot_positions)

pprint({'root': ROOT, 'dataset': dataset, 'sample': sample_id, 'resources_and_outputs': sample_root})

## Step 1: supplied deconvolution and expression reference

Validate and reuse the supplied eight-class deconvolution, then build the platform-normalized reference through the same cerebellum workflow used by `resources/cerebellum`.

In [ ]:
adata = ad.read_h5ad(stage1_st_h5ad, backed='r')
spot_ids = adata.obs_names.astype(str).tolist()
adata.file.close()
deconv = pd.read_csv(deconv_csv, index_col=0)
values = deconv.to_numpy(dtype=np.float64)
if deconv.index.astype(str).tolist() != spot_ids:
    raise ValueError('Deconvolution rows do not match the C2 xDBiT spot order.')
if not np.isfinite(values).all() or np.any(values < 0) or not np.allclose(values.sum(1), 1.0, atol=1e-6):
    raise ValueError('Deconvolution proportions are invalid.')

reference_summary = spacerec.rctd_ref(
    dataset=dataset,
    sample_id=sample_id,
    deconv_csv=deconv_csv,
    stage1_st_h5ad=stage1_st_h5ad,
    stage1_gene_list=stage1_gene_list,
    force=False,
)
gene_list = Path(reference_summary['gene_list'])
mu_ref = Path(reference_summary['mu_ref'])
stage2_st_h5ad = Path(reference_summary['st_h5ad'])
require_files(gene_list, mu_ref, stage2_st_h5ad)
pprint(reference_summary)

## Step 2: OpenPhenom grid embedding and SpaceRec mask

DAPI, phalloidin, and CD31 are fused by channel mean. Local and global 384-dimensional representations are concatenated into each 768-dimensional grid embedding.

In [ ]:
ge_summary = spacerec.ge(
    dataset=dataset,
    sample_id=sample_id,
    model='openphenom',
    he_image=fullres_image,
    positions_csv=spot_positions,
    output_dir=sample_root / 'embedding',
    patch_size=480,
    stride=120,
    batch_size=16,
    force=False,
    enable_progress_bar=True,
)
bbox_embedding = Path(ge_summary['output_h5'])
require_files(bbox_embedding)
pprint(ge_summary)

In [ ]:
mask_summary = spacerec.mask(
    dataset=dataset,
    sample_id=sample_id,
    grid_embedding_h5=bbox_embedding,
    output_dir=sample_root / 'mask',
    force=False,
)
training_embedding = Path(mask_summary['training_h5'])
require_files(training_embedding, Path(mask_summary['mask_h5']))
pprint(mask_summary)

## Step 3.1: Stage 1 router and grid type model

In [ ]:
stage1_summary = spacerec.stage1(
    dataset=dataset,
    sample_id=sample_id,
    st_h5ad=stage1_st_h5ad,
    deconv_csv=deconv_csv,
    grid_embedding_h5=training_embedding,
    gene_list=stage1_gene_list,
    run_dir=sample_root / 'model_pred' / 'train',
    projection_dim=1024,
    stage1_epochs=70,
    batch_size=8,
    skip_input_projector=False,
    factorized_head_hidden_layers=1,
    scale_head_hidden_dim=256,
    delta_head_hidden_dim=256,
    enable_progress_bar=True,
)
stage1_checkpoint = Path(stage1_summary['checkpoint'])
require_files(stage1_checkpoint, Path(stage1_summary['grid_type_csv']))
pprint(stage1_summary)

## Step 3.2: Stage 2 finetune-mu expression model

The validated workflow adapts the Stage 1 checkpoint to the real gene dimension, preserves the trained projector/router, trains the expression heads, and validates the complete grid export.

In [ ]:
stage2_summary = spacerec.stage2(
    dataset=dataset,
    sample_id=sample_id,
    st_h5ad=stage2_st_h5ad,
    deconv_csv=deconv_csv,
    grid_embedding_h5=training_embedding,
    stage1_checkpoint=stage1_checkpoint,
    gene_txt=gene_list,
    mu_ref=mu_ref,
    run_dir=sample_root / 'model_pred' / 'train',
    projection_dim=1024,
    stage2_epochs=70,
    batch_size=4,
    skip_input_projector=False,
    factorized_head_hidden_layers=1,
    scale_head_hidden_dim=256,
    delta_head_hidden_dim=256,
    agg=False,
    enable_progress_bar=True,
)
require_files(Path(stage2_summary['stage2_checkpoint']), Path(stage2_summary['grid_predictions_h5']))
pprint(stage2_summary)

## Step 4: requested C2 window visualization

In [ ]:
from IPython.display import Image, display

bbox_xyxy = (4096, 8192, 8192, 12288)
window_dir = sample_root / 'model_pred' / 'train' / 'metrics' / 'windows' / 'window_4096_8192_8192_12288'
type_plot = spacerec.plottype(
    dataset=dataset,
    grid_type_csv=Path(stage1_summary['grid_type_csv']),
    window=bbox_xyxy,
    output_dir=window_dir,
    output_name='predicted_type.png',
)
expr_plot = spacerec.plotexpr(
    dataset=dataset,
    grid_predictions_h5=Path(stage2_summary['grid_predictions_h5']),
    window=bbox_xyxy,
    output_dir=window_dir,
    gene='Slc17a7',
    output_name='Excitatory_neurons_Slc17a7_grid_expr.png',
)
predicted_type_png = Path(type_plot['grid_type_png'])
slc17a7_png = Path(expr_plot['grid_expr_png'])
require_files(predicted_type_png, slc17a7_png)
display(Image(filename=str(predicted_type_png)))
display(Image(filename=str(slc17a7_png)))